In [3]:
import os
import sqlite3
import dotenv
from pathlib import Path
import pandas as pd

In [4]:
dotenv.load_dotenv()

True

In [5]:
DB_PATH = os.environ["ITX_DATABASE_PATH"]
DB_PATH = str(Path(DB_PATH))  # normaliza

In [6]:
def connect(db_path: str = DB_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row  # para leer por nombre de columna
    return conn

In [7]:
conn = connect()
print("OK, conectado a:", DB_PATH)

OK, conectado a: persistence\backup\interchange.db


In [8]:
def execute(conn: sqlite3.Connection, sql: str, params: tuple = ()) -> int:
    """
    Ejecuta una sentencia (no SELECT). Devuelve rowcount.
    """
    cur = conn.cursor()
    cur.execute(sql, params)
    conn.commit()
    rc = cur.rowcount
    cur.close()
    return rc

def query(conn: sqlite3.Connection, sql: str, params: tuple = ()) -> list[dict]:
    """
    Ejecuta un SELECT. Devuelve lista de dicts.
    """
    cur = conn.cursor()
    cur.execute(sql, params)
    rows = cur.fetchall()
    cur.close()
    return [dict(r) for r in rows]

def executemany(conn: sqlite3.Connection, sql: str, params_list: list[tuple]) -> int:
    """
    Ejecute multiples INSERT/UPDATE/DELETE.
    Devuelve total de filas afectadas.
    """
    cur = conn.cursor()
    cur.executemany(sql, params_list)
    conn.commit()
    rc = cur.rowcount
    cur.close()
    return rc

#### Insertar en Client los bancos 
No se puede insertar aun BT porque sus outgoings tienen bloqueantes y no bloqueantes (VALIDAR)

In [4]:
str_query = """
INSERT INTO client (client_id, client_name, file_mc_block_in, file_mc_block_out, 
file_mc_encoding_in, file_mc_encoding_out, customer_country)
VALUES (?, ?, ?, ?, ?, ?, ?)
"""

In [ ]:
clients = [
    ("ERSTHU", "Erste - Hungary", "TRUE", "TRUE", "Latin-1", "cp500", "HUN"),
    ("BRDRO", "BRD - Romania", "TRUE", "TRUE", "Latin-1", "cp500", "ROM"),
    ("NCBJM", "NCB - Jamaica", "TRUE", "TRUE", "Latin-1", "Latin-1", "JAM"),
    ("BTRLRO", "Bank Transilvania", "TRUE", "TRUE", "Latin-1", "cp500", "ROM")
]

In [46]:
rows = executemany(conn, str_query, clients)
print(f"Filas insertadas: {rows}")

Filas insertadas: 3


#### Insertar en file_control
No se puede insertar aun BT porque sus outgoings tienen bloqueantes y otros no (VALIDAR)

In [16]:
str_query = """
INSERT INTO file_control (
client_id, file_id, brand_id, file_type, landing_file_name, file_processing_date
)
VALUES (?, ?, ?, ?, ?, ?)
"""

In [ ]:
files = [
    ("BRDRO", "ba4a9711221a6b137c56ceb064f54a01", 'MC', "IN", "MI260107.001", '2025-01-20'),
    ("BRDRO", "e88c268de05ae2b194a290031d815d11", 'MC', "IN", "MI260107.004", '2025-01-20'),
    ("BRDRO", "77ddc561c15881790f4e8e86017ae2c0", 'MC', "OUT", "MO260107.001", '2025-01-20'),
    ("BRDRO", "c91be0ff88c4ba42e4faab0fae6a8fbd", 'MC', "OUT", "MO260107.003", '2025-01-20'),
    ("ERSTHU", "2fa77f7c86a64a539974e170208eee9b", "MC", "OUT", "MCI.AR.R111.C.E0085850.D260107.T012940.A001", '2025-01-20'),
    ("ERSTHU", "11eb035fbfadade0da5beb8fa34dd518", "MC", "OUT", "MCI.AR.R111.C.E0085850.D260107.T012958.A002", '2025-01-20'),
    ("ERSTHU", "4b32711bdc8d6c8063dc0021ce67150b", "MC", "IN", "MCI.AR.T112.C.E0085850.D260107.T004239.A003", '2025-01-20'),
    ("ERSTHU", "9c6bbc8152c075cee8b3d14007b6c2e4", "MC", "IN", "MCI.AR.T112.C.E0085850.D260107.T082746.A006", '2025-01-20'),
    ("NCBJM", "ca2dddef155f9698ddf6f43a90a0cd2b", "MC", "OUT", "R06_CLNCBJ_JM_INTERNATIONALTRXNS_031795O_20250716_20250716_225832", '2025-01-20'),
    ("NCBJM", "0db52b5f0fafe6d011675f0492ded440", "MC", "OUT", "R06_CLNCBJ_JM_JMDINTERNATIONALTRXNS_031796O_20250716_20250716_230003", '2025-01-20'),
    ("NCBJM", "55676975578bd419ce037f686329776e", "MC", "OUT", "R06_CLNCBJ_JM_LOCALTRXNS_031792O_20250716_20250716_225706", '2025-01-20'),
    ("NCBJM", "9ae422fdbe1fb4921eaf52201f8bbca7", "MC", "IN", "T11216072530091", '2025-01-20'),
    ("NCBJM", "fc16f6434b2d57528e6f653fbe0d3143", "MC", "IN", "T11216072530092", '2025-01-20'),
    ("NCBJM", "37e637f87b00018ac0364cef8b194060", "MC", "IN", "T11216072530094", '2025-01-20'),
    ("NCBJM", "1fdec35636f3367bdfd5aa176dd6d890", "MC", "IN", "T11216072530096", '2025-01-20'),
    ("NCBJM", "688011e0815a58927f7b0942ab5cfdd3", "MC", "IN", "T11216072555312", '2025-01-20'),
    ("NCBJM", "788f1216e8d32b790fc43fdbe4099352", "MC", "IN", "T11216072555316", '2025-01-20'),
    ("SBSA", "85e91f44241d19d8bf23ce97d2bf49c9", "MC", "IN", "MasterCard_Inward_Settlement_to_SBSA_T112_20260113.TXT", '2025-01-20'),
    ("BTRLRO","a3711894ebf22d0583df63cc5b5232dc","MC", "IN", "MCI.AR.T112.M.E0078853.D260107.T004452.A003", "2025-01-22"),
    ("BTRLRO","3bbe11a245223ecb2ebfb46b6d2c9f36","MC", "IN", "MCI.AR.T112.M.E0078853.D260107.T034734.A004", "2025-01-22"),
    ("BTRLRO","927e539ab0e66cbcf48cd6043cac1d47","MC", "OUT", "IPM_6007.O00063", "2025-01-22"),
    ("BTRLRO","28ef73ae78c526c130fccb618a581359","MC", "OUT", "OMC_20260107_002938_0001", "2025-01-22"),
    ("BTRLRO","cda240036fbee87e93277789a703b8e5","MC", "OUT", "OMC_20260107_022705_0001", "2025-01-22")
]

In [13]:
rows = executemany(conn, str_query, files)
print(f"Filas insertadas: {rows}")

Filas insertadas: 17


### Tabla REGEX

In [ ]:
sql_create = """
CREATE TABLE IF NOT EXISTS file_name_regex_param (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    client_id TEXT NOT NULL,
    brand_id  TEXT NOT NULL CHECK (brand_id IN ('MC', 'VI')),
    file_type TEXT NOT NULL CHECK (file_type IN ('IN', 'OUT')),
    file_format TEXT NOT NULL,
    file_block BOOLEAN NOT NULL,
    creation_date TEXT NOT NULL DEFAULT (datetime('now'))
);
"""

In [13]:
execute(conn, sql_create)

13

In [14]:
sql_insert = """
INSERT INTO file_name_regex_param
(client_id, brand_id, file_type, file_format, file_block)
VALUES (?, ?, ?, ?, ?)
"""

In [ ]:
rows_regex = [
    ("BRDRO", "MC", "IN", r"^MI.*$", "TRUE"),
    ("BRDRO", "MC", "OUT", r"^MO.*$", "TRUE"),
    ("BTRLRO", "MC", "IN", r"^(?:iep_.*\.ipm|early_.*\.ipm|MCI\.AR\.T112\.M.*)$", "TRUE"),
    ("BTRLRO", "MC", "OUT", r"^(?:OMC_.*[^-][^A][^S][^C][^I][^I][^.][^t][^x][^t])$", "FALSE"),
    ("BTRLRO", "MC", "OUT", r"^(?:IPM.*)$", "TRUE"),
    ("ERSTHU", "MC", "IN", r"^MCI.AR.T112.*$", "TRUE"),
    ("ERSTHU", "MC", "OUT", r"^MCI.AR.R111.*$", "TRUE"),
    ("EURBGR", "MC", "IN", r"^(?:T11200A[1-9]d.*|T11200AT0(?:\..*|.*)|T112T0\..*)$", "TRUE"),
    ("EURBGR", "MC", "OUT", r"^(?:TOCEURO.*|ONUSIPM.*)$", "TRUE"),
    ("NCBJM", "MC", "IN", r"^(?:T112|TT112).*", "TRUE"),
    ("NCBJM", "MC", "OUT", r"^(?:MC_)?R06_CLNCBJ_JM_(?:INTERNATIONALTRXNS|JMDINTERNATIONALTRXNS|LOCALTRXNS).*[^-][^A][^S][^C][^I][^I][^.][^t][^x][^t]$", "TRUE"),
    ("SBSA", "MC", "IN", r"^(?:MasterCard_Inward.*)$", "FALSE"),
    ("SBSA", "MC", "OUT", r"(?:Local_MasterCard_Outgoing)|(?:MasterCard_Outward)|(?:CPS)", "FALSE")
]

In [16]:
executemany(conn, sql_insert, rows_regex)

13